# Instacart: limpieza y estandarización de datos

**Evidencia:** GA1-220501094-01-AA1-EV06  
**Lección 2:** Limpieza y estandarización  

Este notebook desarrolla un pipeline reproducible de limpieza con Pandas, genera un informe de cambios y resuelve siete ejercicios prácticos. Se conservan los datos originales en objetos separados y no se imputan los valores ausentes de `days_since_prior_order`, porque son esperables para el primer pedido de cada usuario.

> Los CSV incluidos en este repositorio son datos pequeños de ejemplo. Las conclusiones y conteos de esta ejecución no representan el conjunto completo de Instacart.

## 1. Preparar el entorno y cargar los datos

El bloque localiza la raíz del proyecto tanto si Jupyter se inicia desde `instacart-analisis/` como desde `instacart-analisis/notebooks/`. Los cinco CSV usan punto y coma como separador.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

EXPECTED_FILES = (
    "instacart_orders.csv",
    "order_products.csv",
    "products.csv",
    "aisles.csv",
    "departments.csv",
)

candidate_roots = (Path.cwd(), Path.cwd().parent)
PROJECT_ROOT = next(
    (
        candidate.resolve()
        for candidate in candidate_roots
        if all((candidate / "data" / filename).is_file() for filename in EXPECTED_FILES)
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "No se encontraron los cinco CSV en data/. Inicia Jupyter desde "
        "instacart-analisis/ o desde instacart-analisis/notebooks/."
    )

DATA_DIR = PROJECT_ROOT / "data"
print(f"Raíz del proyecto: {PROJECT_ROOT}")
print(f"Directorio de datos: {DATA_DIR}")

In [ ]:
raw_data = {
    Path(filename).stem: pd.read_csv(DATA_DIR / filename, sep=";")
    for filename in EXPECTED_FILES
}

orders_raw = raw_data["instacart_orders"]
order_products_raw = raw_data["order_products"]
products_raw = raw_data["products"]
aisles_raw = raw_data["aisles"]
departments_raw = raw_data["departments"]

print("Archivos cargados:", ", ".join(raw_data))

## 2. Diagnóstico inicial

Antes de modificar los datos, se revisan dimensiones, tipos, valores ausentes y duplicados. Esta línea base permite explicar qué cambió en el pipeline.

In [ ]:
initial_structure = pd.DataFrame(
    [
        {
            "tabla": name,
            "filas": len(frame),
            "columnas": frame.shape[1],
            "valores_ausentes": int(frame.isna().sum().sum()),
            "filas_duplicadas_exactas": int(frame.duplicated().sum()),
        }
        for name, frame in raw_data.items()
    ]
)
display(initial_structure)

for name, frame in raw_data.items():
    print(f"\n{name}: {frame.shape}")
    display(frame.dtypes.rename("tipo").to_frame())

## 3. Laboratorio guiado 2: pipeline de limpieza con informe de cambios

### Criterios aplicados

1. Se trabaja sobre copias; los DataFrames terminados en `_raw` permanecen intactos.
2. Se eliminan duplicados exactos y filas sin claves requeridas.
3. Los duplicados de clave se eliminan solo si las filas repetidas no tienen atributos contradictorios; ante un conflicto, el pipeline se detiene para evitar descartar información arbitrariamente.
4. Se normalizan espacios y etiquetas de texto, y se convierten columnas numéricas con errores explícitos.
5. Los valores ausentes en `days_since_prior_order` se conservan: representan el primer pedido o la ausencia de historial previo.

In [ ]:
def clean_table(frame, key_columns, table_name):
    """Remove safe duplicates and rows that lack the table's key."""
    original_rows = len(frame)
    result = frame.copy()

    exact_duplicates = int(result.duplicated().sum())
    result = result.drop_duplicates().copy()

    missing_key_mask = result[key_columns].isna().any(axis=1)
    missing_key_rows = int(missing_key_mask.sum())
    result = result.loc[~missing_key_mask].copy()

    duplicate_key_mask = result.duplicated(subset=key_columns, keep=False)
    duplicate_key_rows = result.loc[duplicate_key_mask]
    non_key_columns = [column for column in result.columns if column not in key_columns]
    if not duplicate_key_rows.empty and non_key_columns:
        conflicting_groups = (
            duplicate_key_rows.groupby(key_columns, dropna=False)[non_key_columns]
            .nunique(dropna=False)
            .gt(1)
            .any(axis=1)
        )
        if conflicting_groups.any():
            raise ValueError(
                f"{table_name} tiene claves repetidas con datos contradictorios. "
                "Revisa esos registros antes de continuar."
            )

    duplicate_key_rows_removed = int(
        result.duplicated(subset=key_columns, keep="first").sum()
    )
    result = result.drop_duplicates(subset=key_columns, keep="first").copy()

    change = {
        "tabla": table_name,
        "filas_antes": original_rows,
        "duplicados_exactos_eliminados": exact_duplicates,
        "filas_sin_clave_eliminadas": missing_key_rows,
        "duplicados_de_clave_eliminados": duplicate_key_rows_removed,
        "filas_eliminadas_total": original_rows - len(result),
        "filas_despues": len(result),
        "valores_ausentes_despues": int(result.isna().sum().sum()),
    }
    return result, change

In [ ]:
table_specs = {
    "instacart_orders": (orders_raw, ["order_id"]),
    "order_products": (order_products_raw, ["order_id", "product_id"]),
    "products": (products_raw, ["product_id"]),
    "aisles": (aisles_raw, ["aisle_id"]),
    "departments": (departments_raw, ["department_id"]),
}

cleaned_data = {}
change_records = []
for table_name, (frame, key_columns) in table_specs.items():
    cleaned_data[table_name], change = clean_table(
        frame, key_columns, table_name
    )
    change_records.append(change)

orders_clean = cleaned_data["instacart_orders"]
order_products_clean = cleaned_data["order_products"]
products_clean = cleaned_data["products"]
aisles_clean = cleaned_data["aisles"]
departments_clean = cleaned_data["departments"]

In [ ]:
def normalize_text(frame, columns, lowercase=False):
    for column in columns:
        values = frame[column].astype("string")
        values = values.str.replace(r"\s+", " ", regex=True).str.strip()
        if lowercase:
            values = values.str.lower()
        frame[column] = values


normalize_text(orders_clean, ["eval_set"], lowercase=True)
normalize_text(products_clean, ["product_name"])
normalize_text(aisles_clean, ["aisle"], lowercase=True)
normalize_text(departments_clean, ["department"], lowercase=True)

integer_columns = {
    "instacart_orders": [
        "order_id", "user_id", "order_number", "order_dow", "order_hour_of_day"
    ],
    "order_products": ["order_id", "product_id", "add_to_cart_order", "reordered"],
    "products": ["product_id", "aisle_id", "department_id"],
    "aisles": ["aisle_id"],
    "departments": ["department_id"],
}

for table_name, columns in integer_columns.items():
    for column in columns:
        cleaned_data[table_name][column] = pd.to_numeric(
            cleaned_data[table_name][column], errors="raise"
        ).astype("Int64")

orders_clean["days_since_prior_order"] = pd.to_numeric(
    orders_clean["days_since_prior_order"], errors="raise"
)

print("Texto normalizado y tipos de datos estandarizados.")

In [ ]:
cleaning_report = pd.DataFrame(change_records)
display(cleaning_report)

missing_after = pd.DataFrame(
    [
        {
            "tabla": table_name,
            "columna": column,
            "valores_ausentes": int(count),
        }
        for table_name, frame in cleaned_data.items()
        for column, count in frame.isna().sum().items()
        if count > 0
    ]
)

if missing_after.empty:
    print("No quedan valores ausentes después de limpiar las claves.")
else:
    display(missing_after)

### Lectura del informe

Las filas eliminadas y los valores ausentes restantes quedan cuantificados en el informe. En particular, los valores ausentes de `days_since_prior_order` se mantienen porque no describen un error: el usuario todavía no tiene un pedido anterior registrado. No se reemplazan por cero ni por una media, pues eso alteraría el significado de la variable.

In [ ]:
quality_checks = pd.DataFrame(
    [
        {
            "validacion": "order_id único en pedidos",
            "resultado": orders_clean["order_id"].is_unique,
        },
        {
            "validacion": "product_id único en catálogo",
            "resultado": products_clean["product_id"].is_unique,
        },
        {
            "validacion": "día de semana entre 0 y 6",
            "resultado": orders_clean["order_dow"].between(0, 6).fillna(False).all(),
        },
        {
            "validacion": "hora del día entre 0 y 23",
            "resultado": orders_clean["order_hour_of_day"].between(0, 23).fillna(False).all(),
        },
        {
            "validacion": "reordered contiene solo 0 o 1",
            "resultado": order_products_clean["reordered"].isin([0, 1]).all(),
        },
        {
            "validacion": "productos del pedido existen en el catálogo",
            "resultado": order_products_clean["product_id"].isin(
                products_clean["product_id"]
            ).all(),
        },
        {
            "validacion": "pedidos de order_products existen en orders",
            "resultado": order_products_clean["order_id"].isin(
                orders_clean["order_id"]
            ).all(),
        },
    ]
)
display(quality_checks)

if not quality_checks["resultado"].all():
    raise ValueError("Una o más validaciones de calidad no se cumplieron.")

## 4. Reto 2: siete ejercicios de limpieza

Los ejercicios siguientes practican detección, estandarización y validación usando los cinco DataFrames limpiados. Las soluciones muestran resultados verificables y conservan la trazabilidad de los cambios.

### Ejercicio 1. Cuantificar valores ausentes por tabla y columna

Identifica los campos con datos ausentes después de limpiar las claves y explica por qué no se deben completar automáticamente.

In [ ]:
missing_counts = pd.concat(
    {
        name: frame.isna().sum()
        for name, frame in cleaned_data.items()
    },
    names=["tabla", "columna"],
).rename("valores_ausentes").reset_index()
display(missing_counts.loc[missing_counts["valores_ausentes"].gt(0)])

**Interpretación:** los valores ausentes restantes se inspeccionan por contexto. En `days_since_prior_order`, un valor vacío puede ser correcto para el primer pedido del usuario.

### Ejercicio 2. Detectar duplicados exactos y por clave

Compara duplicados de fila completa con duplicados de la clave definida para cada tabla.

In [ ]:
duplicate_summary = pd.DataFrame(
    [
        {
            "tabla": name,
            "duplicados_exactos_en_original": int(frame.duplicated().sum()),
            "filas_con_clave_repetida_en_original": int(
                frame.duplicated(subset=key_columns, keep=False).sum()
            ),
            "filas_duplicadas_en_limpio": int(
                cleaned_data[name].duplicated(subset=key_columns).sum()
            ),
        }
        for name, (frame, key_columns) in table_specs.items()
    ]
)
display(duplicate_summary)

**Interpretación:** después del pipeline no deben quedar claves repetidas. Si una clave repetida tuviera atributos distintos, la función detendría el proceso para que el conflicto se resuelva explícitamente.

### Ejercicio 3. Normalizar campos de texto

Revisa las etiquetas categóricas y los nombres para confirmar que no queden espacios sobrantes ni diferencias de mayúsculas/minúsculas en las categorías.

In [ ]:
print("eval_set:", sorted(orders_clean["eval_set"].dropna().unique()))
print("departments:", sorted(departments_clean["department"].dropna().unique()))
print("aisles:", sorted(aisles_clean["aisle"].dropna().unique()))
display(products_clean[["product_id", "product_name"]].head())

**Interpretación:** las categorías se guardan en minúsculas y sin espacios laterales; los nombres de productos conservan su capitalización, pero se normalizan los espacios.

### Ejercicio 4. Verificar los tipos de datos

Comprueba los tipos de las columnas de identificadores, cantidades, indicadores y días entre pedidos.

In [ ]:
dtype_summary = pd.concat(
    {name: frame.dtypes.astype(str) for name, frame in cleaned_data.items()},
    names=["tabla", "columna"],
).rename("tipo").reset_index()
display(dtype_summary)

**Interpretación:** las columnas enteras se representan con `Int64` nullable para permitir valores ausentes sin convertir identificadores a números decimales. `days_since_prior_order` conserva su tipo numérico y sus nulos legítimos.

### Ejercicio 5. Contabilizar registros sin claves requeridas

Calcula cuántas filas se eliminaron por tener claves obligatorias ausentes y cuántas filas permanecen en cada tabla.

In [ ]:
key_cleanup = cleaning_report[
    ["tabla", "filas_sin_clave_eliminadas", "filas_despues"]
].sort_values("tabla")
display(key_cleanup)

**Interpretación:** se eliminan únicamente filas que no pueden identificarse de forma confiable. Los faltantes de atributos opcionales no se convierten en motivo automático para descartar el registro.

### Ejercicio 6. Validar reglas de negocio

Busca días y horas fuera de rango, órdenes de carrito no positivas y valores de recompra distintos de 0 o 1.

In [ ]:
invalid_rows = {
    "order_dow_fuera_de_rango": orders_clean.loc[
        ~orders_clean["order_dow"].between(0, 6).fillna(False)
    ],
    "order_hour_fuera_de_rango": orders_clean.loc[
        ~orders_clean["order_hour_of_day"].between(0, 23).fillna(False)
    ],
    "add_to_cart_order_no_positivo": order_products_clean.loc[
        ~order_products_clean["add_to_cart_order"].gt(0).fillna(False)
    ],
    "reordered_no_binario": order_products_clean.loc[
        ~order_products_clean["reordered"].isin([0, 1])
    ],
}

invalid_counts = pd.Series(
    {name: len(invalid_frame) for name, invalid_frame in invalid_rows.items()},
    name="filas_invalidas",
).to_frame()
display(invalid_counts)

**Interpretación:** una validación con cero filas inválidas indica que los registros cumplen la regla revisada; no significa que se hayan eliminado silenciosamente.

### Ejercicio 7. Construir un resumen final de limpieza

Resume el volumen antes/después, las filas removidas y los valores ausentes que permanecen, para dejar evidencia reproducible del pipeline.

In [ ]:
final_summary = cleaning_report[
    [
        "tabla",
        "filas_antes",
        "filas_eliminadas_total",
        "filas_despues",
        "valores_ausentes_despues",
    ]
].copy()
final_summary["porcentaje_filas_conservadas"] = (
    final_summary["filas_despues"]
    .div(final_summary["filas_antes"].where(final_summary["filas_antes"].ne(0)))
    .mul(100)
    .round(2)
)
display(final_summary)

print(
    "El pipeline terminó sin conflictos de claves y todas las validaciones "
    "de calidad incluidas en el laboratorio se cumplieron."
    if quality_checks["resultado"].all()
    else "Revisa las validaciones de calidad antes de usar estos datos."
)

## 5. Conclusiones de la Lección 2

- El pipeline conserva los CSV originales y produce copias limpias en memoria.
- Las claves faltantes y los duplicados se tratan de forma explícita; los conflictos de clave no se resuelven escogiendo registros al azar.
- Las etiquetas de texto y los tipos quedan estandarizados antes de analizar.
- Los valores ausentes tienen que interpretarse según el significado de la columna: un nulo de `days_since_prior_order` puede ser válido.
- El informe de cambios y las validaciones permiten revisar la calidad antes de continuar con filtrado, agrupación y combinación en las siguientes lecciones.